# Week 4 — SQL Intermediate & Advanced (Lakehouse) 
*Catalog/Schema fixed to `bh2026-winford-uc-dev.bh_lakehouse`*
*Each step below is isolated in its own cell to see results per query more clearly.*

-- SQL Lakehouse Design: Bronze, Silver, Gold Layers:

•	bronze_customers
\
•	bronze_products
\
•	bronze_orders
\
•	bronze_order_items
\
•	bronze_payments
\
•	silver_orders_details
\
•	gold_monthly_sales_by_category


## Initialisation & Setup

In [0]:
-- Create Catalog & Schema (explicit)
-- CREATE CATALOG IF NOT EXISTS `bh2026-winford-uc-dev`;
USE CATALOG `bh2026-winford-uc-dev`;
CREATE SCHEMA IF NOT EXISTS bh_lakehouse;
USE SCHEMA bh_lakehouse;
SELECT current_catalog() AS current_catalog, current_schema() AS current_schema;

## 1. CTE — Customer Revenue Summary
Using a CTE, calculate the total amount spent by each customer from the silver_orders_details table.

**Real Analytics Engineering Use Cases**
\
•	KPI dashboards
\
•	Revenue monitoring
\
•	Executive reporting
\
•	Customer health tracking


In [0]:
SELECT *
FROM silver_orders_details;

In [0]:
-- CTE: Customer Revenue Summary
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(payment_amount) AS total_spent
    FROM silver_orders_details
    GROUP BY customer_id
)
SELECT *
FROM customer_revenue;

## Subquery — Above Average Products
Find products with a price higher than the average product price in bronze_products.

**Real Analytics Engineering Use Cases**
\
•	Finance analytics
\
•	Product performance analysis
\
•	Growth analysis
\
•	Forecasting models


In [0]:
-- Products priced above the average product price in bronze_products
SELECT *
FROM bronze_products
WHERE CAST(price AS DOUBLE) > (
    SELECT AVG(CAST(price AS DOUBLE)) FROM bronze_products
);

## ROW_NUMBER() — Customer Order Sequencing
Using bronze_orders, assign a row number to each order per customer ordered by order_date.

**Real Analytics Engineering Use Cases**
\
•	Cohort analysis
\
•	Retention reporting
\
•	Customer journey analytics
\
•	Churn analysis


In [0]:
select * From bronze_products;

In [0]:
-- Assign a row number to each order per customer, ordered by order_date
SELECT
    order_id,
    customer_id,
    order_date,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS order_sequence
FROM bronze_orders;

## RANK() — Highest Spending Customers
Rank customers based on total spending using the silver_orders_details table.

**Real Analytics Engineering Use Cases**
\
•	Executive dashboards
\
•	Revenue monitoring
\
•	Customer segmentation
\
•	Growth analysis


In [0]:
-- Rank customers by total spending (highest first)
SELECT
    customer_id,
    SUM(payment_amount) AS total_spent,
    RANK() OVER (ORDER BY SUM(payment_amount) DESC) AS spending_rank
FROM silver_orders_details
GROUP BY customer_id
ORDER BY spending_rank;

## LAG() — Previous Month Sales Comparison
Using gold_monthly_sales_by_category, calculate the previous month’s sales for each category.

**Real Analytics Engineering Use Cases**
\
•	Forecasting
\
•	Trend analysis
\
•	Revenue monitoring
\
•	Executive reporting


In [0]:
-- LAG: Previous month's sales comparison per category
-- Business Logic: For each product category, compare current month revenue
-- against the prior month to identify growth or decline trends.
WITH monthly_sales AS (
    SELECT
        category,
        DATE_TRUNC('month', order_date) AS sales_month,
        SUM(payment_amount) AS total_sales
    FROM silver_orders_details
    GROUP BY category, DATE_TRUNC('month', order_date)
)
SELECT
    category,
    sales_month,
    total_sales,
    LAG(total_sales) OVER (
        PARTITION BY category
        ORDER BY sales_month
    ) AS prev_month_sales,
    total_sales - LAG(total_sales) OVER (
        PARTITION BY category
        ORDER BY sales_month
    ) AS month_over_month_change
FROM monthly_sales
ORDER BY category, sales_month;

## Running Total — Revenue Growth Tracking
Create a running total of monthly sales using gold_monthly_sales_by_category.

**Real Analytics Engineering Use Cases**
\
•	KPI dashboards
\
•	Revenue growth tracking
\
•	Executive dashboards
\
•	Forecasting models


In [0]:
-- Running Total: Cumulative monthly revenue per category
-- Business Logic: Track how revenue accumulates over time within each category to measure growth trajectory and forecast targets.
WITH monthly_sales AS (
    SELECT
        category,
        DATE_TRUNC('month', order_date) AS sales_month,
        SUM(payment_amount) AS total_sales
    FROM silver_orders_details
    GROUP BY category, DATE_TRUNC('month', order_date)
)
SELECT
    category,
    sales_month,
    total_sales,
    SUM(total_sales) OVER (
        PARTITION BY category
        ORDER BY sales_month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM monthly_sales
ORDER BY category, sales_month;

## CASE WHEN — Product Price Categorisation
Using bronze_products, categorise products as Low Price, Medium Price, or High Price based on product price.

**Real Analytics Engineering Use Cases**
\
•	Customer analytics
\
•	Pricing strategy
\
•	Finance analytics
\
•	Product segmentation


In [0]:
SELECT *,
CASE 
    WHEN CAST(price AS DOUBLE) > 100 THEN 'High Price'
    WHEN CAST(price AS DOUBLE) > 50 THEN 'Medium Price'
    ELSE 'Low Price'
END AS product_price_category
FROM bronze_products;

## Self Join — Customers in the Same City
Using bronze_customers, perform a self-join to find customers living in the same city.

**Real Analytics Engineering Use Cases**
\
•	Customer clustering
\
•	Regional analysis
\
•	Cohort analysis
\
•	Retention analysis


In [0]:
-- Self Join: Find pairs of customers living in the same city
-- Business Logic: Identify customer clusters by geography to support
-- regional marketing campaigns, local event targeting, and cohort analysis.
SELECT
    a.customer_id AS customer_1_id,
    a.customer_name AS customer_1_name,
    b.customer_id AS customer_2_id,
    b.customer_name AS customer_2_name,
    a.city
FROM bronze_customers a
JOIN bronze_customers b
    ON a.city = b.city
    AND a.customer_id < b.customer_id
ORDER BY a.city, a.customer_id;

## Deduplication with ROW_NUMBER()
Using bronze_order_items, remove duplicate records by keeping only the latest row for each order_id and product_id.

**Real Analytics Engineering Use Cases**
\
•	Data quality pipelines
\
•	Silver layer transformations
\
•	Reliable KPI reporting
\
•	Forecasting accuracy

%md
## Silver Layer Transformation Challenge
Using Bronze tables only, create a Silver-style transformation query returning customer_name, order_id, order_date, product_name, category, quantity, unit_price, total_item_price, payment_method, and payment_amount.

**Real Analytics Engineering Use Cases**
\
•	Analytics engineering
\
•	Executive dashboards
\
•	Revenue monitoring
\
•	Forecasting pipelines


In [0]:
-- Deduplication: Keep only the latest row per order_id + product_id
-- Business Logic: In raw (bronze) data, duplicate line items can occur from source system retries or CDC replays. ROW_NUMBER() is used to rank rows within each (order_id, product_id) group and retain only the most recent.
WITH ranked_items AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id, product_id
            ORDER BY order_item_id DESC
        ) AS row_num
    FROM bronze_order_items
)
SELECT
    order_item_id,
    order_id,
    product_id,
    quantity,
    unit_price
FROM ranked_items
WHERE row_num = 1
ORDER BY order_item_id;

In [0]:
-- Silver Layer Transformation: Enrich and join Bronze tables into a single analytics-ready dataset. This is the foundation for Gold aggregations.
-- Business Logic: Combine customer, order, product, and payment data to produce a denormalised fact table for revenue analysis, customer behaviour, and reporting.
SELECT
    c.customer_name,
    o.order_id,
    o.order_date,
    p.product_name,
    p.product_category AS category,
    CAST(oi.quantity AS INT) AS quantity,
    CAST(oi.unit_price AS DOUBLE) AS unit_price,
    CAST(oi.quantity AS INT) * CAST(oi.unit_price AS DOUBLE) AS total_item_price,
    pm.payment_method,
    CAST(pm.amount AS DOUBLE) AS payment_amount
FROM bronze_orders o
JOIN bronze_customers c
    ON o.customer_id = c.customer_id
JOIN bronze_order_items oi
    ON o.order_id = oi.order_id
JOIN bronze_products p
    ON oi.product_id = p.product_id
JOIN bronze_payments pm
    ON o.order_id = pm.order_id
ORDER BY o.order_date, o.order_id;